# ST-Net Benchmark (multi-slide, Protocol A)

Strict ST-Net reproduction using this repository's HEST data and gene list.
Multi-slide supervision: train on two sections, validate on a third, test on the fourth.

Source-faithful core (unchanged from upstream `stnet`, commit `43022c1`):
- `torchvision.models.densenet121(pretrained=True)` -- ImageNet-1k DenseNet-121 encoder
  (`densenet121-a639ec97.pth`, SHA256 a639ec97d7c33b07ae66f0b5fb7d0192f95a3b11b7576c66c0126c2a727c4395)
- `stnet/utils/nn.py::set_out_features` -- classifier replaced by `Linear(1024, G)`, zeroed
- `stnet/transforms/eight_symmetry.py::EightSymmetry` -- the `--average` test-time augmentation
- MSE gene loss `sum((pred - gene)**2) / n_genes`, SGD lr=1e-6 momentum=0.9, batch 32,
  50 epochs, 224 px window at downsample 1 (upstream README "Training models" main command)
- output-layer bias initialised to the training-set mean target (`run_spatial.py`)
- checkpoint chosen by upstream's `bin/cross_validate.py` criterion: run the full budget,
  keep the lowest held-out loss

Adapted data surface only: HEST whole-slide reading, spot-window caching, barcode
alignment, `common_genes.txt` gene panel, fixed slide-level splits, shared log1p(CPM-1e4)
target, prediction export.

See `benchmark/ST-Net/README_MORPHOFM_ADAPTATION.md` for the training-dynamics finding on
the single-slide protocol.


In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Optional OpenSlide
try:
    import openslide
    HAS_OPENSLIDE = True
except ImportError:
    HAS_OPENSLIDE = False
    print("OpenSlide not found, using PIL fallback (may be slow).")

ROOT = Path(os.environ.get("MORPHO_VC_ROOT", "../")).expanduser().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "benchmark") not in sys.path:
    sys.path.insert(0, str(ROOT / "benchmark"))

STNET_ROOT = ROOT / "benchmark" / "ST-Net"

from hest_dataset import (
    build_stnet_model,
    build_stnet_window_cache,
    load_stnet_eight_symmetry,
    stnet_target_matrix,
    HESTSTNetDataset,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# IDs (multi-slide Protocol A defaults; patched per fold by the runner)
train_ids = ["INT25", "INT26"]
val_ids = ["INT27"]
test_ids = ["INT28"]
all_ids = train_ids + val_ids + test_ids

# Paths
result_dir = ROOT / "benchmark" / "results" / "stnet"
result_dir.mkdir(parents=True, exist_ok=True)

window_cache_dir = ROOT / "results" / "prostate_hest" / "stnet" / "windows"
window_cache_dir.mkdir(parents=True, exist_ok=True)

spatial_dir = ROOT / "data" / "spatial_data"
wsi_dir = ROOT / "data" / "hest_data" / "wsis"
common_gene_path = spatial_dir / "common_genes.txt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ST-Net window convention (upstream stnet/datasets/spatial.py at --downsample 1):
# a 224 px full-resolution crop with no rescale -- see README_MORPHOFM_ADAPTATION.md.
STNET_WINDOW = 224
STNET_CROP_MODE = "tile_px"
NUM_WORKERS = 0

print("Result dir:", result_dir)
print(f"Train: {train_ids} | Val: {val_ids} | Test: {test_ids}")
print("Device:", device)


In [ ]:
# ===== Helpers =====


def read_gene_list(path: Path):
    genes = path.read_text(encoding="utf-8").splitlines()
    return [g.strip() for g in genes if g.strip()]


print("Helpers ready.")


In [ ]:
# ===== Step 1: cached 224 px H&E windows, one per in-tissue spot =====

genes = read_gene_list(common_gene_path)
print(f"Genes: {len(genes)}")

window_bundle = build_stnet_window_cache(
    all_ids,
    hest_dir=ROOT / "data" / "hest_data",
    spatial_dir=spatial_dir,
    window=STNET_WINDOW,
    crop_mode=STNET_CROP_MODE,
    cache_dir=window_cache_dir,
    verbose=True,
)

for sid in all_ids:
    print(f"{sid}: windows {window_bundle[sid]['shape']} | cache {window_bundle[sid]['cache_path']}")


In [ ]:
# ===== Step 2: Targets + train/valid/test datasets =====

import anndata as ad

targets = {}
for sid in all_ids:
    adata = ad.read_h5ad(spatial_dir / f"{sid}.h5ad")
    targets[sid] = stnet_target_matrix(adata, genes)
    print(f"{sid}: target {targets[sid].shape}")

train_set = HESTSTNetDataset({s: window_bundle[s] for s in train_ids},
                             {s: targets[s] for s in train_ids}, genes)
valid_set = HESTSTNetDataset({s: window_bundle[s] for s in val_ids},
                             {s: targets[s] for s in val_ids}, genes)
test_set = HESTSTNetDataset({s: window_bundle[s] for s in test_ids},
                            {s: targets[s] for s in test_ids}, genes)
print(f"train_set: {len(train_set)} | valid_set: {len(valid_set)} | test_set: {len(test_set)}")


In [ ]:
# ===== Step 3: Train upstream ST-Net (source-faithful) =====
#
# Upstream `run_spatial.py` at the README main command: densenet121, window 224,
# pretrain, average, batch 32, lr 1e-6, SGD momentum 0.9, 50 epochs. Only the data
# surface (HEST h5ad + WSI windows, common_genes panel, shared log1p(CPM-1e4) target,
# fixed slide-level splits) and the checkpoint selection differ.
#
# Checkpoint selection: upstream `bin/cross_validate.py` runs the full epoch budget and
# keeps the epoch with the lowest held-out loss. This template does the same against the
# held-out validation slide, which is what the other Protocol-A baselines also monitor
# (`ModelCheckpoint(save_top_k=1, monitor='valid_loss')`).

import copy as _copy
import random

BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 1e-6
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0
USE_AVERAGE = True  # upstream --average
OUTPUTS = len(genes)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Upstream mean/std estimate (run_spatial.py): first 11 shuffled training batches ---
train_set.transform = torchvision.transforms.ToTensor()
est_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
mean = 0.0
std = 0.0
n = 0
for i, (X, _y) in enumerate(est_loader):
    X = X.transpose(0, 1).contiguous().view(3, -1)
    n += X.shape[1]
    mean = mean + torch.sum(X, dim=1)
    std = std + torch.sum(X ** 2, dim=1)
    if i > 10:  # upstream `if i > n_samples: break` with n_samples = 10
        break
mean = mean / n
std = torch.sqrt(std / n - mean ** 2)
print("channel mean:", mean, "channel std:", std)

# --- Upstream transforms for the --average setting ---
train_transform = torchvision.transforms.Compose([
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.RandomVerticalFlip(),
    torchvision.transforms.RandomApply([torchvision.transforms.RandomRotation((90, 90))]),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=mean, std=std),
])
train_set.transform = train_transform

EightSymmetry = load_stnet_eight_symmetry(STNET_ROOT)
eval_transform = torchvision.transforms.Compose([
    EightSymmetry(),
    torchvision.transforms.Lambda(
        lambda symmetries: torch.stack(
            [torchvision.transforms.Normalize(mean=mean, std=std)(
                torchvision.transforms.ToTensor()(s)) for s in symmetries]
        )
    ),
])
valid_set.transform = eval_transform
test_set.transform = eval_transform

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

# --- Model: torchvision DenseNet-121 + upstream set_out_features ---
model = build_stnet_model(STNET_ROOT, OUTPUTS, pretrained=True, device=device)

# Upstream initialises the output bias to the mean target of the training set
# (mean_expression in run_spatial.py; weight is already zeroed by set_out_features).
mean_expression = torch.zeros(OUTPUTS)
_n_seen = 0
for _sid in train_ids:
    _y = np.asarray(targets[_sid], dtype=np.float32)
    mean_expression += torch.from_numpy(_y.sum(axis=0))
    _n_seen += _y.shape[0]
mean_expression /= _n_seen
last = model.classifier
last.weight.data.zero_()
last.bias.data = mean_expression.clone().to(device)

optim = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE,
                        momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)


def run_epoch(model, optim, loader, train: bool):
    """One pass of upstream's run_spatial.py inner loop.

    Returns the quantity upstream logs as `Loss`, which for `--gene` with no
    `--gene_mask` is `sum_batches sum((pred - gene)**2) / n_genes` divided by the
    number of samples -- i.e. the plain MSE over all spots and genes.
    """
    torch.set_grad_enabled(train)
    model.train(train)
    total = 0.0
    for (X, y) in loader:
        X = X.to(device)
        y = y.to(device)
        if not train and USE_AVERAGE:
            batch, n_sym, c, h, w = X.shape
            X = X.view(-1, c, h, w)
        pred = model(X)
        if not train and USE_AVERAGE:
            pred = pred.view(batch, n_sym, -1).mean(1)
        loss = torch.sum((pred - y) ** 2) / OUTPUTS
        if train:
            optim.zero_grad()
            loss.backward()
            optim.step()
        total += float(loss.detach().cpu())
    return total / len(loader.dataset)


train_loss, valid_loss = [], []
best_state = None
best_val = float("inf")
best_epoch = -1

for epoch in range(EPOCHS):
    tl = run_epoch(model, optim, train_loader, train=True)
    vl = run_epoch(model, optim, valid_loader, train=False)
    train_loss.append(tl)
    valid_loss.append(vl)
    flag = ""
    if vl < best_val:
        best_val = vl
        best_epoch = epoch
        best_state = _copy.deepcopy(model.state_dict())
        flag = "  <- best"
    print(f"epoch {epoch + 1}/{EPOCHS} | train_loss {tl:.6f} | valid_loss {vl:.6f}{flag}")

model.load_state_dict(best_state)
print(f"\nBest epoch {best_epoch + 1} | valid MSE {best_val:.6f} (restored)")

torch.save(model.state_dict(), result_dir / "model_trained.pth")
np.savetxt(result_dir / "train_valid_loss.txt",
           np.array((train_loss, valid_loss)).T, fmt="%.6f")
with open(result_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "method": "ST-Net",
        "protocol": "rotating_heldout_multi_slide",
        "train_slide_ids": train_ids, "val_slide_ids": val_ids, "test_slide_ids": test_ids,
        "model": "densenet121", "pretrained": True,
        "window": STNET_WINDOW, "crop_mode": STNET_CROP_MODE,
        "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
        "optimizer": "SGD", "average": USE_AVERAGE, "seed": SEED,
        "best_epoch": best_epoch + 1, "best_valid_mse": float(best_val),
        "channel_mean": [float(v) for v in mean], "channel_std": [float(v) for v in std],
        "n_genes": len(genes),
    }, f, ensure_ascii=False, indent=2)
print("Saved:", result_dir / "model_trained.pth")


In [ ]:
# ===== Step 4: Evaluation + Save pred_bag/true_bag + metrics =====

model.eval()

pred_bag_list = []
true_bag_list = []

with torch.no_grad():
    for sid in test_ids:
        base = HESTSTNetDataset({sid: window_bundle[sid]}, {sid: targets[sid]}, genes,
                                transform=eval_transform)
        loader = DataLoader(base, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        preds = []
        for X, _y in loader:
            X = X.to(device)
            batch, n_sym, c, h, w = X.shape
            X = X.view(-1, c, h, w)
            p = model(X)
            p = p.view(batch, n_sym, -1).mean(1)
            preds.append(p.detach().cpu().numpy())
        pred_vals = np.concatenate(preds, axis=0)      # (n_spots, n_genes)
        true_vals = targets[sid]                       # (n_spots, n_genes)
        pred_bag_list.append(pred_vals)
        true_bag_list.append(true_vals)
        print(f"{sid}: pred {pred_vals.shape} | true {true_vals.shape}")

pred_bag = np.concatenate(pred_bag_list, axis=0).astype(np.float32)
true_bag = np.concatenate(true_bag_list, axis=0).astype(np.float32)
print("Prediction Shape:", pred_bag.shape)


def pearson_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if a.size == 0 or b.size == 0:
        return np.nan
    if np.all(a == a[0]) or np.all(b == b[0]):
        return np.nan
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt((a * a).sum()) * np.sqrt((b * b).sum())
    if denom == 0:
        return np.nan
    return float((a * b).sum() / denom)


mae = float(np.mean(np.abs(pred_bag - true_bag)))
rmse = float(np.sqrt(np.mean((pred_bag - true_bag) ** 2)))

corrs = [pearson_corr(pred_bag[:, i], true_bag[:, i]) for i in range(pred_bag.shape[1])]
valid = [(i, c) for i, c in enumerate(corrs) if not np.isnan(c)]
mean_pearson = float(np.mean([c for _, c in valid])) if valid else float("nan")
best_gene_idx, best_gene_corr = max(valid, key=lambda x: x[1]) if valid else (None, None)
best_gene_name = genes[best_gene_idx] if best_gene_idx is not None else "NA"

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Mean Pearson (gene-wise): {mean_pearson:.4f}")
print(f"Best Gene: {best_gene_name} (Pearson={best_gene_corr:.4f})")

np.save(result_dir / "pred_bag.npy", pred_bag)
np.save(result_dir / "true_bag.npy", true_bag)
with open(result_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "MAE": mae, "RMSE": rmse, "Mean_Pearson": mean_pearson,
        "Best_Gene": best_gene_name,
        "Best_Pearson": float(best_gene_corr) if best_gene_corr is not None else None,
    }, f, ensure_ascii=False, indent=2)
print("Saved pred_bag/true_bag/metrics.json")


In [ ]:
# ===== Step 5: Visualization (01-style, top-4 genes on the test slide) =====

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

VIS_ID = test_ids[0]
adata_vis = ad.read_h5ad(spatial_dir / f"{VIS_ID}.h5ad")
mask_vis = adata_vis.obs["in_tissue"].to_numpy().astype(bool)
xy = adata_vis.obs[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy()[mask_vis]

pred_vals = pred_bag[: mask_vis.sum()]
true_vals = true_bag[: mask_vis.sum()]
corrs_vis = np.array([pearson_corr(pred_vals[:, i], true_vals[:, i]) for i in range(pred_vals.shape[1])])
valid_idx = np.where(np.isfinite(corrs_vis))[0]
top4 = valid_idx[np.argsort(corrs_vis[valid_idx])[::-1][:4]]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for j, gi in enumerate(top4):
    for row, (vals, tag) in enumerate([(true_vals, "measured"), (pred_vals, "predicted")]):
        ax = axes[row, j]
        sc = ax.scatter(xy[:, 0], xy[:, 1], c=vals[:, gi], s=3, cmap="viridis")
        ax.set_title(f"{genes[gi]} ({tag})\nr={corrs_vis[gi]:.3f}")
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(sc, ax=ax, fraction=0.046)
plt.tight_layout()
out_fig = result_dir / "top4_genes_vis.png"
plt.savefig(out_fig, dpi=150)
plt.close()
print("Saved:", out_fig)
